# Climate Intelligence Engine

Pipeline analítico para detecção de inconsistências, completamento de dados e otimização de precisão climática.

Execute cada célula em sequência para rodar o pipeline completo de pré-processamento.

In [ ]:
import subprocess
import sys

def run(script):
    print(f"▶  {script}\n{'─' * 50}")
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{script} falhou (código {result.returncode})")
    print("✓  Concluído\n")

## 1.1 — Download dos Dados

Baixa `stations.parquet` e `weather_measurements.parquet` do Google Drive para `data/`.

In [ ]:
run("1.1_download_data.py")

## 1.2 — Cálculo de Distâncias entre Estações

Gera `data/station_distances.parquet` com distâncias par-a-par (Haversine + altitude) entre as 663 estações.

> Requer: 1.1

In [ ]:
run("1.2_compute_station_distances.py")

## 1.3 — Análise Exploratória

Executa o notebook de EDA e exporta o relatório PDF para `doc/1.3_exploratory_analysis.pdf`.

> Requer: 1.1

In [ ]:
# print("▶  1.3_exploratory_analysis.ipynb\n" + "─" * 50)
# result = subprocess.run(
#     ["jupyter", "nbconvert", "--to", "notebook", "--execute",
#      "--inplace", "--ExecutePreprocessor.timeout=600",
#      "1.3_exploratory_analysis.ipynb"],
#     capture_output=True, text=True,
# )
# print(result.stdout or result.stderr)
# if result.returncode != 0:
#     raise RuntimeError(f"1.3 falhou (código {result.returncode})")
# print("✓  Concluído\n")

## 1.4 — Limpeza e Separação das Variáveis

Gera um Parquet por variável (`temperature`, `humidity`, `rainfall`, `global_radiation`, `pressure`) com schema `code | time | measurement`.

> Requer: 1.1

In [ ]:
run("1.4_clean_data.py")

## 1.5 — Enriquecimento com Vizinhos

Para cada variável, enriquece cada medição com os 20 vizinhos mais próximos disponíveis no mesmo timestamp.

Schema de saída (107 colunas): `code | time | measurement | n01..n20 | d01..d20 | a01..a20 | b01_sin/cos..b20_sin/cos | hour_sin/cos | doy_sin/cos`

> Requer: 1.1, 1.2, 1.4

In [ ]:
run("1.5_build_neighbors.py")

## 1.6 — Normalização das Features (StandardScaler)

Aplica StandardScaler (µ=0, σ=1) em todas as colunas numéricas dos arquivos `_neighbors.parquet`. Salva o parquet escalado em `data/` e o scaler em `models/1.6_scaler_{variable}.json`.

> Requer: 1.5

In [ ]:
run("1.6_scale_features.py")

## 2.0 — Neighbors (Baseline: Vizinho Mais Próximo)

Avalia o vizinho mais próximo (n01) como estimador da medição real no conjunto de teste. Gera `results/2.0_neighbors/metrics.csv` com MAE, RMSE, R², Bias e r por variável.

> Requer: 1.6

In [ ]:
run("2.0_neighbors.py")

## 3.0 — Regressão Linear

Treina OLS no conjunto de treino e avalia no teste. Salva `results/3.0_linear_regression/{variable}/model.npy` e `metrics.csv` por variável.

> Requer: 1.6

In [ ]:
run("3.0_linear_regression.py")

## Análise — Cobertura de vizinhos por estação/ano

Para cada ano, mostra as 10 estações com **menos vizinhos disponíveis** (mediana de slots não-NaN em n01..n20).
Serve para calibrar `START_YEAR` no 1.6 — escolher o menor ano com cobertura aceitável.

> Requer: 1.5 (usa `temperature_neighbors.parquet` como proxy)

In [1]:
from pathlib import Path
import pandas as pd

K          = 20
TOP_N      = 10
ncols      = [f"n{i+1:02d}" for i in range(K)]

df = pd.read_parquet(
    Path("data") / "temperature_neighbors.parquet",
    columns=["code", "time"] + ncols,
)
df["year"]        = pd.to_datetime(df["time"]).dt.year
df["n_available"] = df[ncols].notna().sum(axis=1)

by_station_year = (
    df.groupby(["year", "code"])["n_available"]
    .median()
    .reset_index()
    .rename(columns={"n_available": "med_neighbors"})
)

for year in sorted(by_station_year["year"].unique()):
    worst = (
        by_station_year[by_station_year["year"] == year]
        .nsmallest(TOP_N, "med_neighbors")
    )
    print(f"station {year}")
    for _, row in worst.iterrows():
        print(f"  {row['code']:<20}  {row['med_neighbors']:>4.0f}")
    print()

station 2000
  A001                     2
  A101                     2
  A601                     2
  A401                     4
  A801                     4

station 2001
  A001                     6
  A101                     6
  A401                     6
  A601                     6
  A801                     6
  A002                     9
  A003                     9
  A005                     9
  A702                    11
  A703                    11

station 2002
  A001                    15
  A002                    15
  A003                    15
  A401                    15
  A601                    15
  A702                    15
  A703                    15
  A704                    15
  A801                    15
  A802                    15

station 2003
  A001                    20
  A002                    20
  A003                    20
  A101                    20
  A201                    20
  A202                    20
  A203                    20
  A303           

In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(Path("data") / "temperature_train_scaled.parquet")
print(df.shape)
df.head(10)

(53375043, 82)


,code,time,measurement,n01,n02,n03,n04,n05,n06,n07,...,b10_cos,b11_cos,b12_cos,b13_cos,b14_cos,b15_cos,hour_sin,hour_cos,doy_sin,doy_cos
0,A001,2002-01-01,0.382900,0.525926,0.557196,0.535055,0.592593,0.553114,0.522222,0.608856,...,-0.918782,-0.980809,-0.940210,-0.918738,-0.976037,0.714482,0.0,1.0,0.017213,0.999852
1,A601,2002-01-01,0.605948,0.548148,0.527675,0.594096,0.388889,0.465201,0.403704,0.523985,...,0.885768,-0.022150,-0.641173,-0.798079,-0.641588,0.752438,0.0,1.0,0.017213,0.999852
2,A801,2002-01-01,0.457249,0.559259,0.583026,0.409594,0.481481,0.553114,0.466667,0.590406,...,0.987154,0.976226,0.992713,0.945826,0.796288,0.945863,0.0,1.0,0.017213,0.999852
3,A101,2002-01-01,0.639405,0.555556,0.531365,0.391144,0.525926,0.531136,0.588889,0.468635,...,-0.793725,-0.974802,-0.978180,-0.989086,-0.959607,-0.972688,0.0,1.0,0.017213,0.999852
4,A703,2002-01-01,0.464684,0.522222,0.590406,0.409594,0.551852,0.586081,0.525926,0.479705,...,-0.953608,-0.058954,0.816491,0.693348,0.459436,0.975488,0.0,1.0,0.017213,0.999852
5,A402,2002-01-01,0.527881,0.555556,0.387454,0.579336,0.529630,0.611722,0.588889,0.549816,...,0.501404,-0.896934,-0.957800,-0.918297,-0.901480,-0.955859,0.0,1.0,0.017213,0.999852
6,A003,2002-01-01,0.524164,0.385185,0.590406,0.560886,0.551852,0.527473,0.529630,0.608856,...,0.397278,-0.989402,-0.948061,-0.923259,-0.983701,0.794384,0.0,1.0,0.017213,0.999852
7,A802,2002-01-01,0.557621,0.459259,0.583026,0.483395,0.407407,0.472527,0.548148,0.590406,...,0.979362,0.968984,0.987239,0.940773,0.804886,0.961860,0.0,1.0,0.017213,0.999852
8,A803,2002-01-01,0.579926,0.477778,0.405904,0.464945,0.562963,0.472527,0.548148,0.590406,...,0.937175,0.926049,0.963493,0.895178,0.727020,0.971048,0.0,1.0,0.017213,0.999852
9,A005,2002-01-01,0.553903,0.385185,0.531365,0.531365,0.592593,0.527473,0.548148,0.575646,...,0.675677,-0.961432,-0.994240,-0.971039,-0.954453,-0.990341,0.0,1.0,0.017213,0.999852


In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(Path("data") / "temperature_train_scaled.parquet", columns=["time"])

print("dtype       :", df["time"].dtype)
print("min         :", df["time"].min())
print("max         :", df["time"].max())
print("valores únicos de hora (sample):", sorted(df["time"].dt.hour.unique())[:12])
print("\nregistros com hora == 0 :", (df["time"].dt.hour == 0).sum())
print("registros com hora != 0 :", (df["time"].dt.hour != 0).sum())
print("\nprimeiras 5:")
print(df["time"].head())
print("\núltimas 5:")
print(df["time"].tail())